# Logistic regression — companion notebook

Companion for [`logistic-regression.md`](./logistic-regression.md). Fit by gradient descent (own loop) and sklearn; visualize the decision boundary on 2D synthetic data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)

In [ ]:
# Two Gaussian blobs
n = 300
X0 = rng.multivariate_normal([-1.0, -0.5], [[1.0, 0.2],[0.2, 1.0]], size=n//2)
X1 = rng.multivariate_normal([ 1.2,  1.0], [[1.0,-0.3],[-0.3,1.0]], size=n//2)
X = np.vstack([X0, X1])
y = np.array([0]*len(X0) + [1]*len(X1))
X_aug = np.hstack([X, np.ones((n,1))])  # add bias column

def sigmoid(z): return 1/(1+np.exp(-z))

# Hand-rolled GD on negative log-likelihood
w = np.zeros(3); lr = 0.05
losses = []
for it in range(1500):
    p = sigmoid(X_aug @ w)
    grad = X_aug.T @ (p - y) / n
    w -= lr * grad
    losses.append(-np.mean(y*np.log(p+1e-12) + (1-y)*np.log(1-p+1e-12)))
print(f'GD weights: w={w[:2]}, b={w[2]:.3f}')

clf = LogisticRegression(C=1e6).fit(X, y)
print(f'sklearn:    w={clf.coef_[0]}, b={clf.intercept_[0]:.3f}')

In [ ]:
# Decision boundary visualization
gx, gy = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
Z = sigmoid(np.c_[gx.ravel(), gy.ravel(), np.ones(gx.size)] @ w).reshape(gx.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ax1, ax2 = axes
cs = ax1.contourf(gx, gy, Z, levels=20, cmap='RdBu_r', alpha=0.6)
ax1.contour(gx, gy, Z, levels=[0.5], colors='black')
ax1.scatter(X[y==0,0], X[y==0,1], c='blue', s=10); ax1.scatter(X[y==1,0], X[y==1,1], c='red', s=10)
ax1.set_title('Logistic-regression P(y=1|x) and decision boundary'); ax1.set_aspect('equal')

ax2.plot(losses); ax2.set_xlabel('GD iteration'); ax2.set_ylabel('binary cross-entropy'); ax2.set_title('Training loss')
plt.show()